# Teste Groq API — geração de exemplos N1

Notebook autónomo para:

1. ligar à Groq API;
2. listar modelos disponíveis;
3. selecionar um modelo compatível;
4. gerar 2 exemplos N1 em JSON;
5. validar e descarregar o resultado.

No Colab, cria um Secret chamado `GROQ_API_KEY`.

In [1]:
# ============================================================
# CONFIGURAÇÃO
# ============================================================

NUMBER_OF_EXAMPLES = 2

OUTPUT_DIR = "/content/groq_outputs"
OUTPUT_FILE = f"{OUTPUT_DIR}/n1_groq_test_{NUMBER_OF_EXAMPLES}_examples.json"

DOMAIN = "Bibliotecas"
FACT_TYPE = "Capacidade"
ENTITY_TYPE = "Biblioteca"
CONTEXT_LENGTH = "curto, com 3 a 5 frases"
FACT_POSITION = "no fim do contexto"
QUESTION_TYPE = "pergunta direta"
DISTRACTOR_RULE = "não incluir informação enganadora nem distratores relevantes"

MODEL_CANDIDATES = [
    "llama-3.3-70b-versatile",
    "openai/gpt-oss-120b",
    "openai/gpt-oss-20b",
    "qwen/qwen3-32b",
    "moonshotai/kimi-k2-instruct-0905",
    "meta-llama/llama-4-scout-17b-16e-instruct",
]

TEMPERATURE = 0.8
MAX_TOKENS = 4000

In [2]:
!pip -q install -U groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 5.3 MB/s eta 0:00:00


In [3]:
import json
import re
from pathlib import Path

from groq import Groq
from google.colab import userdata

api_key = userdata.get("GROQ_API_KEY")

if not api_key:
    raise ValueError("Adiciona GROQ_API_KEY aos Secrets do Colab.")

client = Groq(api_key=api_key)
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print("Cliente Groq criado.")

Cliente Groq criado.


In [4]:
models_response = client.models.list()

available_models = sorted(model.id for model in models_response.data)

print(f"Modelos encontrados: {len(available_models)}\n")

for model_id in available_models:
    print(model_id)

Modelos encontrados: 17

allam-2-7b
canopylabs/orpheus-arabic-saudi
canopylabs/orpheus-v1-english
groq/compound
groq/compound-mini
llama-3.1-8b-instant
llama-3.3-70b-versatile
meta-llama/llama-4-scout-17b-16e-instruct
meta-llama/llama-prompt-guard-2-22m
meta-llama/llama-prompt-guard-2-86m
openai/gpt-oss-120b
openai/gpt-oss-20b
openai/gpt-oss-safeguard-20b
qwen/qwen3-32b
qwen/qwen3.6-27b
whisper-large-v3
whisper-large-v3-turbo


In [5]:
MODEL_NAME = next(
    (m for m in MODEL_CANDIDATES if m in available_models),
    None,
)

if MODEL_NAME is None:
    raise RuntimeError(
        "Nenhum modelo candidato está disponível. "
        "Adiciona à lista MODEL_CANDIDATES um modelo apresentado acima."
    )

print("Modelo selecionado:", MODEL_NAME)

Modelo selecionado: llama-3.3-70b-versatile


In [6]:
test_completion = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "user", "content": "Responde apenas com a palavra: Olá"}
    ],
    temperature=0,
    max_tokens=20,
)

print(test_completion.choices[0].message.content)

Olá


In [7]:
prompt = f"""
Gera exatamente {NUMBER_OF_EXAMPLES} exemplos em português europeu.

OBJETIVO
Criar exemplos para treinar um modelo pequeno a localizar um facto explícito
num contexto fornecido.

DISTRIBUIÇÃO
Todos os exemplos devem ter estas características:
- domínio: {DOMAIN}
- tipo de facto pedido: {FACT_TYPE}
- tipo de entidade principal: {ENTITY_TYPE}
- contexto: {CONTEXT_LENGTH}
- posição do facto que responde à pergunta: {FACT_POSITION}
- tipo de pergunta: {QUESTION_TYPE}
- distratores: {DISTRACTOR_RULE}

REGRAS
1. Cada exemplo deve conter exatamente:
   - context
   - question
   - answer
2. A resposta deve estar claramente escrita no contexto.
3. Deve existir apenas uma resposta correta.
4. A pergunta deve pedir apenas um facto.
5. Não uses conhecimento externo.
6. Usa entidades realistas, preferencialmente inventadas.
7. Não repitas exemplos.
8. Devolve apenas JSON válido.
9. Não uses Markdown.
10. Não acrescentes explicações.

FORMATO OBRIGATÓRIO
{{
  "examples": [
    {{
      "context": "...",
      "question": "...",
      "answer": "..."
    }}
  ]
}}
""".strip()

print(prompt)

Gera exatamente 2 exemplos em português europeu.

OBJETIVO
Criar exemplos para treinar um modelo pequeno a localizar um facto explícito
num contexto fornecido.

DISTRIBUIÇÃO
Todos os exemplos devem ter estas características:
- domínio: Bibliotecas
- tipo de facto pedido: Capacidade
- tipo de entidade principal: Biblioteca
- contexto: curto, com 3 a 5 frases
- posição do facto que responde à pergunta: no fim do contexto
- tipo de pergunta: pergunta direta
- distratores: não incluir informação enganadora nem distratores relevantes

REGRAS
1. Cada exemplo deve conter exatamente:
   - context
   - question
   - answer
2. A resposta deve estar claramente escrita no contexto.
3. Deve existir apenas uma resposta correta.
4. A pergunta deve pedir apenas um facto.
5. Não uses conhecimento externo.
6. Usa entidades realistas, preferencialmente inventadas.
7. Não repitas exemplos.
8. Devolve apenas JSON válido.
9. Não uses Markdown.
10. Não acrescentes explicações.

FORMATO OBRIGATÓRIO
{
  "examp

In [8]:
completion = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {
            "role": "system",
            "content": "Devolve apenas JSON válido e segue rigorosamente a instrução."
        },
        {
            "role": "user",
            "content": prompt
        }
    ],
    temperature=TEMPERATURE,
    max_tokens=MAX_TOKENS,
    response_format={"type": "json_object"},
)

raw_text = completion.choices[0].message.content.strip()

print(raw_text[:2000])

{
  "examples": [
       {
           "context": "A Biblioteca Municipal de São Pedro foi inaugurada em 2001. Ela oferece uma variedade de serviços, incluindo empréstimo de livros, acesso à internet e salas de estudo. A biblioteca tem uma capacidade para 500 pessoas.",
           "question": "Qual é a capacidade da Biblioteca Municipal de São Pedro?",
           "answer": "500 pessoas"
       },
       {
           "context": "A Biblioteca Central de Lisboa é uma das maiores do país. Ela abriga mais de 100 mil volumes e recebe diariamente centenas de visitantes. A biblioteca foi projetada para ter uma capacidade de 800 pessoas.",
           "question": "Qual é a capacidade da Biblioteca Central de Lisboa?",
           "answer": "800 pessoas"
       }
   ]
}


In [9]:
def clean_json_text(text):
    text = text.strip()

    if text.startswith("```json"):
        text = text[len("```json"):]

    if text.startswith("```"):
        text = text[len("```"):]

    if text.endswith("```"):
        text = text[:-3]

    return text.strip()


payload = json.loads(clean_json_text(raw_text))

if not isinstance(payload, dict) or "examples" not in payload:
    raise ValueError("A resposta deve ser um objeto com a chave 'examples'.")

examples = payload["examples"]

if not isinstance(examples, list):
    raise TypeError("'examples' deve ser uma lista.")

if len(examples) != NUMBER_OF_EXAMPLES:
    raise ValueError(
        f"Esperava {NUMBER_OF_EXAMPLES} exemplos, recebi {len(examples)}."
    )

required = {"context", "question", "answer"}

for i, example in enumerate(examples, start=1):
    missing = required - set(example.keys())

    if missing:
        raise ValueError(f"Exemplo {i}: campos em falta: {sorted(missing)}")

print(f"JSON válido: {len(examples)} exemplos.")

JSON válido: 2 exemplos.


In [10]:
def normalize(text):
    text = re.sub(r"\s+", " ", str(text).lower().strip())
    return text.strip(" .,:;!?")


for i, example in enumerate(examples, start=1):
    answer_in_context = (
        normalize(example["answer"]) in normalize(example["context"])
    )
    print(f"Exemplo {i}: resposta no contexto = {answer_in_context}")

Exemplo 1: resposta no contexto = True
Exemplo 2: resposta no contexto = True


In [12]:
for i, example in enumerate(examples, start=1):
    print("\n" + "=" * 80)
    print("EXEMPLO", i)
    print("Contexto:", example["context"])
    print("Pergunta:", example["question"])
    print("Resposta:", example["answer"])


EXEMPLO 1
Contexto: A Biblioteca Municipal de São Pedro foi inaugurada em 2001. Ela oferece uma variedade de serviços, incluindo empréstimo de livros, acesso à internet e salas de estudo. A biblioteca tem uma capacidade para 500 pessoas.
Pergunta: Qual é a capacidade da Biblioteca Municipal de São Pedro?
Resposta: 500 pessoas

EXEMPLO 2
Contexto: A Biblioteca Central de Lisboa é uma das maiores do país. Ela abriga mais de 100 mil volumes e recebe diariamente centenas de visitantes. A biblioteca foi projetada para ter uma capacidade de 800 pessoas.
Pergunta: Qual é a capacidade da Biblioteca Central de Lisboa?
Resposta: 800 pessoas


In [13]:
result = {
    "provider": "groq",
    "model": MODEL_NAME,
    "number_of_examples": len(examples),
    "generation_settings": {
        "temperature": TEMPERATURE,
        "domain": DOMAIN,
        "fact_type": FACT_TYPE,
        "entity_type": ENTITY_TYPE,
        "context_length": CONTEXT_LENGTH,
        "fact_position": FACT_POSITION,
        "question_type": QUESTION_TYPE,
    },
    "examples": examples,
}

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

print("Guardado em:", OUTPUT_FILE)

Guardado em: /content/groq_outputs/n1_groq_test_2_examples.json


In [14]:
from google.colab import files
files.download(OUTPUT_FILE)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Depois de validares os 2 exemplos, altera `NUMBER_OF_EXAMPLES = 2` para `20` e executa novamente.